# Comparing and choosing a regression model

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Which model should I choose?
No model is always best (the **no free lunch theorem**). The right approach: compare several models with **the same cross-validation**.

> [Mean 5-fold R² on the diabetes data. On this small, nearly linear data set, linear models win.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/reg_compare.png)

*Mean 5-fold R² on the diabetes data. On this small, nearly linear data set, linear models win.*

Interesting: on the diabetes data the simple linear models beat forests and boosting. The data is small (442 samples) and the relation is nearly linear, so complex models only learn more noise.

## A guide table
| Model | Needs scaling? | Non-linear? | Interpretation | Training speed | Big data |
|---|---|---|---|---|---|
| Linear | No | No | Excellent | Very fast | Excellent |
| Ridge / Lasso / ENet | **Yes** | No | Excellent | Very fast | Excellent |
| Polynomial | **Yes** | Yes | Medium | Fast | Features explode |
| KNN | **Yes** | Yes | Low | Instant (slow prediction) | Poor |
| Tree | No | Yes | Excellent | Fast | Good |
| Random forest | No | Yes | Medium | Medium | Good |
| Gradient boosting | No | Yes | Medium | Medium | Excellent (Hist) |
| SVR | **Yes** | Yes | Low | Slow | Poor |

## A practical recipe
- Always start with a **baseline**: `DummyRegressor` (the mean) and linear regression.
- If the relation is non-linear, try a random forest (little tuning).
- For the best accuracy on tabular data, gradient boosting with early stopping.
- If interpretation matters, Lasso or a shallow tree.
- Test the final model **once** on the test data.

[[lab:reg_all]]

In the lab, try every model on the "steps" and "with outliers" data. Which model is best on the step data? Why?



In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.model_selection import KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

X, y = load_diabetes(return_X_y=True)
models = {
    "Dummy (mean)": DummyRegressor(),
    "Linear": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV()),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(random_state=0)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor(15)),
    "Tree": DecisionTreeRegressor(max_depth=3, random_state=0),
    "Forest": RandomForestRegressor(200, min_samples_leaf=5, random_state=0),
    "Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, random_state=0),
    "SVR": make_pipeline(StandardScaler(), SVR(C=100, epsilon=10)),
}
cv = KFold(5, shuffle=True, random_state=0)   # the same folds for every model
rows = []
for name, m in models.items():
    r = cross_validate(m, X, y, cv=cv, scoring=["r2", "neg_mean_absolute_error"])
    rows.append([name, r["test_r2"].mean(), -r["test_neg_mean_absolute_error"].mean(), r["fit_time"].mean() * 1000])
print(pd.DataFrame(rows, columns=["model", "R2", "MAE", "fit ms"]).sort_values("R2", ascending=False).round(3).to_string(index=False))



## Predicted against actual
A good diagnostic: if the points lie on the line $y=x$, the model is good. Regular patterns in the residuals mean the model missed something.



In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict

X, y = load_diabetes(return_X_y=True)
pred = cross_val_predict(LinearRegression(), X, y, cv=5)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(y, pred, s=12); ax[0].plot([25, 346], [25, 346], "r--")
ax[0].set(xlabel="actual", ylabel="predicted", title="Predicted vs actual")
ax[1].scatter(pred, y - pred, s=12); ax[1].axhline(0, color="r", ls="--")
ax[1].set(xlabel="predicted", ylabel="residual", title="Residuals")
plt.tight_layout(); plt.show()